# Notebook 01: Data Ingestion & Schema Validation
## Purpose
Load all 4 CMS datasets into Databricks DataFrames, validate schemas,
profile data quality, and document findings before any transformation begins.

## Inputs
- cms_inpatient_provider_service.csv — 146k rows, Medicare inpatient by hospital & DRG
- cms_inpatient_by_provider.csv — Hospital-level payment summaries
- cms_hospital_general_info.csv — Hospital names, addresses, type, ratings
- cms_complications_deaths.csv — Surgical complication rates by hospital

## Outputs
- 4 validated DataFrames ready for transformation in Notebook 02
- Data quality report documenting nulls, row counts, and anomalies

## Author
Chandra Bajepally
## Date
March 2026

In [0]:
# Import all libraries we need
import pandas as pd
import numpy as np

print("Libraries loaded successfully")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

Libraries loaded successfully
Pandas version: 2.2.3
NumPy version: 2.1.3


In [0]:
# Load dataset 1 — Provider and Service

df_provider_service = pd.read_csv(
    "/Volumes/workspace/default/raw_data/cms_inpatient_provider_service.csv",
    dtype=str,
    encoding ='latin-1' # Load everything as string first — safer for initial load
)

print("Loaded data from Provider and Service ")
print(f"Rows: {len(df_provider_service):,}")
print(f"Columns: {len(df_provider_service.columns)}")
print(f"\nColumn Names:")
for col in df_provider_service.columns:
    print(f"  - {col}")

Loaded data from Provider and Service 
Rows: 146,427
Columns: 15

Column Names:
  - Rndrng_Prvdr_CCN
  - Rndrng_Prvdr_Org_Name
  - Rndrng_Prvdr_City
  - Rndrng_Prvdr_St
  - Rndrng_Prvdr_State_FIPS
  - Rndrng_Prvdr_Zip5
  - Rndrng_Prvdr_State_Abrvtn
  - Rndrng_Prvdr_RUCA
  - Rndrng_Prvdr_RUCA_Desc
  - DRG_Cd
  - DRG_Desc
  - Tot_Dschrgs
  - Avg_Submtd_Cvrd_Chrg
  - Avg_Tot_Pymt_Amt
  - Avg_Mdcr_Pymt_Amt


In [0]:
# Load dataset 2 — By Provider

df_by_provider = pd.read_csv(
    "/Volumes/workspace/default/raw_data/cms_inpatient_by_provider.csv",
    dtype=str,
    encoding ='latin-1' # Load everything as string first — safer for initial load
)

print("Loaded data from Provider ")
print(f"Rows: {len(df_by_provider):,}")
print(f"Columns: {len(df_by_provider.columns)}")
print(f"\nColumn Names:")
for col in df_by_provider.columns:
    print(f"  - {col}")

Loaded data from Provider 
Rows: 3,093
Columns: 57

Column Names:
  - Rndrng_Prvdr_CCN
  - Rndrng_Prvdr_Org_Name
  - Rndrng_Prvdr_St
  - Rndrng_Prvdr_City
  - Rndrng_Prvdr_Zip5
  - Rndrng_Prvdr_State_Abrvtn
  - Rndrng_Prvdr_State_FIPS
  - Rndrng_Prvdr_RUCA
  - Rndrng_Prvdr_RUCA_Desc
  - Tot_Benes
  - Tot_Submtd_Cvrd_Chrg
  - Tot_Pymt_Amt
  - Tot_Mdcr_Pymt_Amt
  - Tot_Dschrgs
  - Tot_Cvrd_Days
  - Tot_Days
  - Bene_Avg_Age
  - Bene_Age_LT_65_Cnt
  - Bene_Age_65_74_Cnt
  - Bene_Age_75_84_Cnt
  - Bene_Age_GT_84_Cnt
  - Bene_Feml_Cnt
  - Bene_Male_Cnt
  - Bene_Race_Wht_Cnt
  - Bene_Race_Black_Cnt
  - Bene_Race_API_Cnt
  - Bene_Race_Hspnc_Cnt
  - Bene_Race_NatInd_Cnt
  - Bene_Race_Othr_Cnt
  - Bene_Dual_Cnt
  - Bene_Ndual_Cnt
  - Bene_CC_BH_ADHD_OthCD_V1_Pct
  - Bene_CC_BH_Alcohol_Drug_V1_Pct
  - Bene_CC_BH_Tobacco_V1_Pct
  - Bene_CC_BH_Alz_NonAlzdem_V2_Pct
  - Bene_CC_BH_Anxiety_V1_Pct
  - Bene_CC_BH_Bipolar_V1_Pct
  - Bene_CC_BH_Mood_V2_Pct
  - Bene_CC_BH_Depress_V1_Pct
  - Bene_CC_BH_PD_

In [0]:
# Load dataset 3 - Hospital General Info

df_hospital_info = pd.read_csv(
    "/Volumes/workspace/default/raw_data/cms_hospital_general_info.csv",
    dtype=str,
    encoding ='latin-1' # Load everything as string first — safer for initial load
)

print("Loaded data from Hospital General Info ")
print(f"Rows: {len(df_hospital_info):,}")
print(f"Columns: {len(df_hospital_info.columns)}")
print(f"\nColumn Names:")
for col in df_hospital_info.columns:
    print(f"  - {col}")

Loaded data from Hospital General Info 
Rows: 5,426
Columns: 38

Column Names:
  - Facility ID
  - Facility Name
  - Address
  - City/Town
  - State
  - ZIP Code
  - County/Parish
  - Telephone Number
  - Hospital Type
  - Hospital Ownership
  - Emergency Services
  - Meets criteria for birthing friendly designation
  - Hospital overall rating
  - Hospital overall rating footnote
  - MORT Group Measure Count
  - Count of Facility MORT Measures
  - Count of MORT Measures Better
  - Count of MORT Measures No Different
  - Count of MORT Measures Worse
  - MORT Group Footnote
  - Safety Group Measure Count
  - Count of Facility Safety Measures
  - Count of Safety Measures Better
  - Count of Safety Measures No Different
  - Count of Safety Measures Worse
  - Safety Group Footnote
  - READM Group Measure Count
  - Count of Facility READM Measures
  - Count of READM Measures Better
  - Count of READM Measures No Different
  - Count of READM Measures Worse
  - READM Group Footnote
  - Pt Exp 

In [0]:
# Load dataset 4 - Complications and Deaths

df_complications = pd.read_csv(
    "/Volumes/workspace/default/raw_data/cms_complications_deaths.csv",
    dtype=str,
    encoding ='latin-1' # Load everything as string first — safer for initial load
)

print("Loaded data from Complications and Deaths ")
print(f"Rows: {len(df_complications):,}")
print(f"Columns: {len(df_complications.columns)}")
print(f"\nColumn Names:")
for col in df_complications.columns:
    print(f"  - {col}")

Loaded data from Complications and Deaths 
Rows: 95,780
Columns: 18

Column Names:
  - Facility ID
  - Facility Name
  - Address
  - City/Town
  - State
  - ZIP Code
  - County/Parish
  - Telephone Number
  - Measure ID
  - Measure Name
  - Compared to National
  - Denominator
  - Score
  - Lower Estimate
  - Higher Estimate
  - Footnote
  - Start Date
  - End Date


In [0]:
# For each dataset check for null counts, data types and sample rows 
# Building a Quality report

def data_quality_report(df, name):
    print(f"\n{'='*60}")
    print(f"DATA QUALITY REPORT: {name}")
    print(f"{'='*60}")
    print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
    
    # Null analysis
    null_counts = df.isnull().sum()
    null_pct = (null_counts / len(df) * 100).round(2)
    null_report = pd.DataFrame({
        'Null Count': null_counts,
        'Null %': null_pct
    })
    
    # Only show columns that have nulls
    nulls_found = null_report[null_report['Null Count'] > 0]
    if len(nulls_found) > 0:
        print(f"\nColumns with nulls:")
        print(nulls_found.to_string())
    else:
        print(f"\n✅ No null values found")
    
    # Sample data
    print(f"\nFirst 3 rows:")
    print(df.head(3).to_string())

# Run report for all 4 datasets
data_quality_report(df_provider_service, "Provider and Service")
data_quality_report(df_by_provider, "By Provider")
data_quality_report(df_hospital_info, "Hospital General Info")
data_quality_report(df_complications, "Complications and Deaths")


DATA QUALITY REPORT: Provider and Service
Shape: 146,427 rows x 15 columns

Columns with nulls:
                        Null Count  Null %
Rndrng_Prvdr_RUCA                4     0.0
Rndrng_Prvdr_RUCA_Desc           4     0.0

First 3 rows:
  Rndrng_Prvdr_CCN            Rndrng_Prvdr_Org_Name Rndrng_Prvdr_City         Rndrng_Prvdr_St Rndrng_Prvdr_State_FIPS Rndrng_Prvdr_Zip5 Rndrng_Prvdr_State_Abrvtn Rndrng_Prvdr_RUCA                                                                                Rndrng_Prvdr_RUCA_Desc DRG_Cd                                                                                  DRG_Desc Tot_Dschrgs Avg_Submtd_Cvrd_Chrg Avg_Tot_Pymt_Amt Avg_Mdcr_Pymt_Amt
0           010001  Southeast Health Medical Center            Dothan  1108 Ross Clark Circle                      01             36301                        AL                 2  Metropolitan area high commuting: primary flow 30% or more to a urbanized area of 50,000 and greater    003  ECMO OR TRACHEOSTOMY W

In [0]:
# Confirm how all 4 datasets connect to each other
# The hospital ID (CCN) is our primary join key

print("=== JOIN KEY ANALYSIS ===\n")

# Dataset 1 & 2 use: Rndrng_Prvdr_CCN
print("Dataset 1 - CCN sample values:")
print(df_provider_service['Rndrng_Prvdr_CCN'].head(3).tolist())

print("\nDataset 2 - CCN sample values:")
print(df_by_provider['Rndrng_Prvdr_CCN'].head(3).tolist())

# Dataset 3 & 4 use: Facility ID
print("\nDataset 3 - Facility ID sample values:")
print(df_hospital_info['Facility ID'].head(3).tolist())

print("\nDataset 4 - Facility ID sample values:")
print(df_complications['Facility ID'].head(3).tolist())

print("\n Join Strategy:")
print("  Datasets 1+2: Join on Rndrng_Prvdr_CCN")
print("  Datasets 3+4: Join on Facility ID")
print("  Cross join:   Rndrng_Prvdr_CCN = Facility ID (same values, different names)")

=== JOIN KEY ANALYSIS ===

Dataset 1 - CCN sample values:
['010001', '010001', '010001']

Dataset 2 - CCN sample values:
['010001', '010005', '010006']

Dataset 3 - Facility ID sample values:
['010001', '010005', '010006']

Dataset 4 - Facility ID sample values:
['010001', '010001', '010001']

 Join Strategy:
  Datasets 1+2: Join on Rndrng_Prvdr_CCN
  Datasets 3+4: Join on Facility ID
  Cross join:   Rndrng_Prvdr_CCN = Facility ID (same values, different names)


In [0]:
# HOSPITAL COVERAGE VALIDATION
# How many unique hospitals are in each dataset?


print("=== UNIQUE HOSPITAL COUNTS ===\n")

d1_hospitals = df_provider_service['Rndrng_Prvdr_CCN'].nunique()
d2_hospitals = df_by_provider['Rndrng_Prvdr_CCN'].nunique()
d3_hospitals = df_hospital_info['Facility ID'].nunique()
d4_hospitals = df_complications['Facility ID'].nunique()

print(f"Dataset 1 (Provider & Service): {d1_hospitals:,} unique hospitals")
print(f"Dataset 2 (By Provider):        {d2_hospitals:,} unique hospitals")
print(f"Dataset 3 (Hospital Info):      {d3_hospitals:,} unique hospitals")
print(f"Dataset 4 (Complications):      {d4_hospitals:,} unique hospitals")

# Check overlap between Dataset 1 and Dataset 3
d1_ids = set(df_provider_service['Rndrng_Prvdr_CCN'].unique())
d3_ids = set(df_hospital_info['Facility ID'].unique())
overlap = d1_ids.intersection(d3_ids)

print(f"\nHospitals in BOTH Dataset 1 & Dataset 3: {len(overlap):,}")
print(f"This is our joinable universe of hospitals")

=== UNIQUE HOSPITAL COUNTS ===

Dataset 1 (Provider & Service): 2,945 unique hospitals
Dataset 2 (By Provider):        3,093 unique hospitals
Dataset 3 (Hospital Info):      5,426 unique hospitals
Dataset 4 (Complications):      4,789 unique hospitals

Hospitals in BOTH Dataset 1 & Dataset 3: 2,880
This is our joinable universe of hospitals


In [0]:
# SURGICAL DRG PREVIEW
# Preview the types of procedures in our data

print("=== SAMPLE DRG PROCEDURES IN DATASET 1 ===\n")

# Show unique DRG descriptions — sample of 20
drg_sample = df_provider_service[['DRG_Cd', 'DRG_Desc']]\
    .drop_duplicates()\
    .sort_values('DRG_Cd')\
    .head(20)

for _, row in drg_sample.iterrows():
    print(f"  DRG {row['DRG_Cd']}: {row['DRG_Desc'][:70]}")

print(f"\nTotal unique DRG codes: {df_provider_service['DRG_Cd'].nunique()}")
print("\nIn Notebook 02 we will filter these to surgical procedures only")
print("(orthopedic, cardiac, spinal, general surgery categories)")

=== SAMPLE DRG PROCEDURES IN DATASET 1 ===

  DRG 001: HEART TRANSPLANT OR IMPLANT OF HEART ASSIST SYSTEM WITH MCC
  DRG 003: ECMO OR TRACHEOSTOMY WITH MV >96 HOURS OR PRINCIPAL DIAGNOSIS EXCEPT F
  DRG 004: TRACHEOSTOMY WITH MV >96 HOURS OR PRINCIPAL DIAGNOSIS EXCEPT FACE, MOU
  DRG 005: LIVER TRANSPLANT WITH MCC OR INTESTINAL TRANSPLANT
  DRG 006: LIVER TRANSPLANT WITHOUT MCC
  DRG 007: LUNG TRANSPLANT
  DRG 011: TRACHEOSTOMY FOR FACE, MOUTH AND NECK DIAGNOSES OR LARYNGECTOMY WITH M
  DRG 012: TRACHEOSTOMY FOR FACE, MOUTH AND NECK DIAGNOSES OR LARYNGECTOMY WITH C
  DRG 013: TRACHEOSTOMY FOR FACE, MOUTH AND NECK DIAGNOSES OR LARYNGECTOMY WITHOU
  DRG 014: ALLOGENEIC BONE MARROW TRANSPLANT
  DRG 016: AUTOLOGOUS BONE MARROW TRANSPLANT WITH CC/MCC
  DRG 018: CHIMERIC ANTIGEN RECEPTOR (CAR) T-CELL AND OTHER IMMUNOTHERAPIES
  DRG 020: INTRACRANIAL VASCULAR PROCEDURES WITH PRINCIPAL DIAGNOSIS HEMORRHAGE W
  DRG 023: CRANIOTOMY WITH MAJOR DEVICE IMPLANT OR ACUTE COMPLEX CNS PRINCIPAL DI
  DR